# Bronze → Silver | CineData Analytics

A camada **Silver** é onde o dado bruto vira dado **confiável**:

| O que fiz | Por quê |
|---|---|
| Renomear colunas para português | Padronização |
| Converter tipos | Na Bronze tudo estava como `STRING`; aqui damos o tipo correto |
| Limpar sujeira  | Sem isso a conversão de tipo falha ou vira `NULL` sem querer |
| Remover duplicados | A Bronze é `append`: rodar 2x duplica linhas |
| Aplicar regras de negócio | Dado fora da regra é dado errado, não pode seguir para análise |

**Regras gerais deste notebook**

1. As tabelas Silver são gravadas com `overwrite`: a cada execução a Silver é reconstruída do zero a partir
   de toda a Bronze. Isso deixa o pipeline idempotente.
2. Conversão segura de tipo (`try_cast`): o compute serverless roda em modo ANSI, onde um `cast` comum
   derruba o notebook ao encontrar texto como "abc" numa coluna numérica. Com `try_cast`/`try_to_date`,
   o valor impossível de converter vira `NULL` e o pipeline segue.

## 0. Configurações e funções auxiliares

In [0]:
from datetime import date

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "workspace"
BANCO_BRONZE = "bronze"
BANCO_SILVER = "silver"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BANCO_SILVER}")   # banco da camada Silver

DataFrame[]

In [0]:
# CONSTANTES DE NEGÓCIO

# Textos que significam não temos esse dado. Antes de converter para número, esses textos precisam virar NULL.
TOKENS_AUSENCIA = [
    "unknown", "não informado", "nao informado", "n/a", "na", "nan", "null", "none",
    "-", "--", "desconhecido", "undefined", "",
]

# Sobras que sobram quando uma lista é separada por vírgula, ex.: "Downey, Jr" que vira "Downey" e "Jr."
SUFIXOS_SOLTOS = ["inc", "inc.", "ltd", "ltd.", "llc", "co", "co.", "corp", "corp.", "jr", "jr.", "sr", "sr."]

### Funções auxiliares

Coloquei todas aqui em cima para o resto do notebook ficar mais curto e fácil de entender.

In [0]:
# LEITURA DA BRONZE
def ler_bronze(tabela: str):
    """
    Lê uma tabela da Bronze e cria a coluna auxiliar _ordem_linha.

    Por que _ordem_linha? Todas as linhas carregadas numa mesma execução da Bronze têm o MESMO
    ingestion_datetime, o current_timestamp() é calculado uma vez por gravação. Se houver duplicados
    dentro do mesmo arquivo, precisamos de um critério de desempate então usei a posição da linha 
    a linha que aparece mais abaixo no arquivo é considerada a mais nova.
    """
    return spark.table(f"{BANCO_BRONZE}.{tabela}").withColumn("_ordem_linha", F.monotonically_increasing_id())


def manter_versao_mais_recente(df, chaves):
    """
    Deduplicação: para cada chave mantém só a versão mais recente,
    ordenando por ingestion_datetime decrescente e, no empate, pela posição da linha.

    row_number() numera as linhas de cada grupo ficando com a de número 1.
    """
    janela = Window.partitionBy(*chaves).orderBy(F.col("ingestion_datetime").desc(), F.col("_ordem_linha").desc())
    return df.withColumn("_rn", F.row_number().over(janela)).filter(F.col("_rn") == 1).drop("_rn")


def limpar_id(df, coluna="id"):
    """Remove espaços do id e descarta linhas sem id pois sem chave não dá para ligar as tabelas."""
    return (
        df.withColumn(coluna, F.trim(F.col(coluna)))
          .filter(F.col(coluna).isNotNull() & (F.col(coluna) != ""))
    )


# LIMPEZA DE TEXTO
def limpar_texto(coluna: str):
    """
    Tira espaços das pontas e transforma texto vazio em NULL. Retorna uma Column.
    """
    aparado = F.regexp_replace(F.col(coluna), r"^[\s\u00a0]+|[\s\u00a0]+$", "")
    return F.when(aparado == "", F.lit(None)).otherwise(aparado)


# LIMPEZA DE NÚMEROS (todas devolvem TEXTO limpo, pronto para o try_cast)
def limpar_valor_monetario(coluna: str):
    """
    Para orçamento/receita. Um ponto/vírgula seguido de exatamente 3 dígitos é separador de MILHAR.
    """
    bruto = F.lower(F.trim(F.col(coluna)))
    ausente = bruto.isNull() | bruto.isin(TOKENS_AUSENCIA)

    notacao_cientifica = bruto.rlike(r"^-?\d+(\.\d+)?e\+?\d+$")

    # Remove tudo que não for dígito, ponto, vírgula ou sinal de menos (símbolos de moeda, letras, espaços)
    s = F.regexp_replace(bruto, r"[^0-9.,\-]", "")

    formato_br = s.rlike(r"^-?\d{1,3}(\.\d{3})+(,\d{1,2})?$")   # 1.500.000,50
    formato_us = s.rlike(r"^-?\d{1,3}(,\d{3})+(\.\d{1,2})?$")   # 1,500,000.50

    limpo = (
        F.when(notacao_cientifica, bruto)
         .when(formato_br, F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", "."))
         .when(formato_us, F.regexp_replace(s, ",", ""))
         .otherwise(F.regexp_replace(s, ",", "."))         
    )
    return F.when(ausente, F.lit(None)).otherwise(limpo)


def limpar_decimal(coluna: str):
    # tira aspas e pontuação solta nas pontas: '"7.5"' -> '7.5', '7.5.' -> '7.5'
    s = F.regexp_replace(F.trim(F.col(coluna)), r"^[\s'\".,;:]+|[\s'\".,;:]+$", "")
    parece_numero = s.rlike(r"^-?\d[\d.,]*$")

    ultimo_sep = F.regexp_extract(s, r"([.,])[^.,]*$", 1)                        # último "." ou "," do texto
    qtd_ponto = F.length(s) - F.length(F.regexp_replace(s, r"\.", ""))
    qtd_virgula = F.length(s) - F.length(F.regexp_replace(s, ",", ""))

    virgula_e_decimal = (ultimo_sep == ",") & (qtd_virgula == 1)
    ponto_e_decimal = (ultimo_sep == ".") & (qtd_ponto == 1)

    limpo = (
        F.when(virgula_e_decimal, F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", "."))
         .when(ponto_e_decimal, F.regexp_replace(s, ",", ""))
         .otherwise(F.regexp_replace(s, r"[.,]", ""))
    )
    return F.when(parece_numero, limpo).otherwise(F.lit(None))


def limpar_inteiro(coluna: str):
    """
    Para contagens de votos. Aceita "1234", "1,234", "1.234" e "1234.0".
    Qualquer outro texto (Column Shift) vira NULL.
    """
    s = F.trim(F.col(coluna))
    com_milhar = s.rlike(r"^-?\d{1,3}([.,]\d{3})+$")
    decimal_zero = s.rlike(r"^-?\d+[.,]0+$")
    simples = s.rlike(r"^-?\d+$")
    return (
        F.when(com_milhar, F.regexp_replace(s, r"[.,]", ""))
         .when(decimal_zero, F.regexp_replace(s, r"[.,]0+$", ""))
         .when(simples, s)
         .otherwise(F.lit(None))
    )


def converter_numero(df, origem: str, destino: str, limpador, tipo: str):
    """
    Aplica o limpador e depois a conversão fica com o tipo (try_cast).
    Se não der para converter, o resultado é NULL.
    Para inteiros passamos antes por double, porque o try_cast não converte direto para int.
    """
    df = df.withColumn("_tmp", limpador(origem))
    if tipo in ("int", "bigint"):
        conversao = f"try_cast(try_cast(`_tmp` as double) as {tipo})"
    else:
        conversao = f"try_cast(`_tmp` as {tipo})"
    return df.withColumn(destino, F.expr(conversao)).drop("_tmp")


# LISTAS SEPARADAS POR VÍRGULA / PONTO E VÍRGULA
def dividir_lista(coluna: str):
    """
    A origem guarda vários valores na mesma célula.
    1) remove colchetes e aspas duplas (resto de listas no formato ["a", "b"]);
    2) divide por vírgula, ponto e vírgula ou barra vertical, ignorando espaços em volta.
    Devolve um array de textos, pronto para explode().
    """
    sem_lixo = F.regexp_replace(F.col(coluna), r"[\[\]\"]", "")
    return F.split(sem_lixo, r"\s*[,;|]\s*")


# GRAVAÇÃO E RESUMO
def salvar_silver(df, tabela: str) -> None:
    """Grava em Delta com overwrite (reconstrói a tabela inteira a cada execução)."""
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BANCO_SILVER}.{tabela}")
    )
    print(f"OK  {BANCO_SILVER}.{tabela}  |  {spark.table(f'{BANCO_SILVER}.{tabela}').count()} linhas")


def resumo_nulos(df):
    """Mostra quantos NULLs existem em cada coluna (bom para conferir se a limpeza não 'comeu' dados demais)."""
    display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]))

## 1. `silver.tb_cotacao_dolar`

**O problema:** a API do Banco Central não devolve cotação em sábados, domingos e feriados. Se somarmos valores
por data, esses dias "somem".

**A solução:**
1. Ficar com uma cotação por dia (a API devolve vários boletins no mesmo dia; usando o último, o de fechamento).
2. Criar um calendário contínuo (todos os dias entre a primeira e a última data).
3. Forward Fill: dia sem cotação recebe o valor do último dia que tinha.

In [0]:
df = ler_bronze("tb_cotacao_dolar") # rodar o outro notebook senão da erro

if df.limit(1).count() == 0:
    raise ValueError("bronze.tb_cotacao_dolar está vazia. Rode o notebook Landing_to_Bronze primeiro.")

# dataHoraCotacao vem como "2026-09-15 13:06:52.155"; os 10 primeiros caracteres são a data.
df = (
    df.withColumn("data_cotacao", F.expr("try_to_date(substring(`dataHoraCotacao`, 1, 10), 'yyyy-MM-dd')"))
      .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("double"))
      .filter(F.col("data_cotacao").isNotNull() & (F.col("cotacao_compra") > 0))
)

# 1) Uma cotação por dia: a mais tardia do dia. No empate, a ingestão mais recente.
janela_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc(), F.col("_ordem_linha").desc()
)
cotacao_diaria = (
    df.withColumn("_rn", F.row_number().over(janela_dia))
      .filter(F.col("_rn") == 1)
      .select("data_cotacao", "cotacao_compra")
)

# 2) Calendário contínuo: uma linha para cada dia entre a menor e a maior data com cotação
limites = cotacao_diaria.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim")).first()
calendario = spark.range(1).select(
    F.explode(F.sequence(F.lit(limites["ini"]), F.lit(limites["fim"]), F.expr("interval 1 day"))).alias("data_cotacao")
)

# 3) Forward Fill: left join traz NULL nos dias sem cotação; last(ignorenulls=True) pega o último valor
janela_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

serie_continua = (
    calendario.join(cotacao_diaria.withColumnRenamed("cotacao_compra", "cotacao_original"), "data_cotacao", "left")
    .withColumn("cotacao_compra", F.last("cotacao_original", ignorenulls=True).over(janela_ffill))
    .withColumn("cotacao_preenchida", F.col("cotacao_original").isNull())   # True = dia sem cotação
    .select("data_cotacao", "cotacao_compra", "cotacao_preenchida")
)

salvar_silver(serie_continua, "tb_cotacao_dolar")
display(spark.table(f"{BANCO_SILVER}.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK  silver.tb_cotacao_dolar  |  11 linhas


data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-21,5.1111,false
2026-09-20,5.1569,true
2026-09-19,5.1569,true
2026-09-18,5.1569,false
2026-09-17,5.1515,false
2026-09-16,5.152,false
2026-09-15,5.1484,false
2026-09-14,5.169,false
2026-09-13,5.0912,true
2026-09-12,5.0912,true


## 2. `silver.tb_info_filmes`  (origem: `tb_movies_info`)

Ordem dos passos:
1. **Deduplicar primeiro**: assim limpamos só a versão que vale.
2. **Normalizar e traduzir o status**.
3. **Converter a data** testando vários formatos.
4. Renomear colunas e criar `ano_lancamento`.

In [0]:
df = ler_bronze("tb_movies_info")

# Passo 1: deduplicação
# se o mesmo id aparece mais de uma vez, fica só a versão com a data de ingestão mais recente.
df = limpar_id(df, "id")
df = manter_versao_mais_recente(df, ["id"])

# Passo 2: status 
# Normalização: tudo minúsculos + remove o que não for letra.
# Depois tudo o que não estiver no mapa (lixo, NULL, valores corrompidos) vira "Não Informado".
MAPA_STATUS = {
    "released":       "Lançado",
    "postproduction": "Pós-Produção",
    "inproduction":   "Em Produção",
    "planned":        "Planejado",
    "rumored":        "Rumores",
    "canceled":       "Cancelado",
    "cancelled":      "Cancelado",     
}
status_normalizado = F.regexp_replace(F.lower(F.trim(F.col("status"))), r"[^a-z]", "")

status_traduzido = F.lit("Não Informado")                      
for termo_origem, termo_pt in MAPA_STATUS.items():
    status_traduzido = F.when(status_normalizado == termo_origem, F.lit(termo_pt)).otherwise(status_traduzido)

# Passo 3: data de lançamento em vários formatos 
# a) Se vier com hora fica só com a parte da data.
df = df.withColumn(
    "_data_txt",
    F.regexp_replace(F.trim(F.col("release_date")), r"^(\d{4}-\d{1,2}-\d{1,2})[T ].*$", "$1"),
)

# b) Testamos cada formato com try_to_date (NULL se o formato não bate) e o coalesce pega o
#     primeiro que funcionou. Letras únicas (d, M) aceitam 1 ou 2 dígitos ("5/3/2020" e "05/03/2020").
#     (dia/mês) primeiro depois datas como "12/25/2020" (mês 25 não existe) caem naturalmente no formato americano.
FORMATOS_DATA = [
    "yyyy-M-d", "d/M/yyyy", "M/d/yyyy", "yyyy/M/d", "d-M-yyyy", "M-d-yyyy",
    "d.M.yyyy", "yyyyMMdd", "d MMM yyyy", "MMM d, yyyy", "MMMM d, yyyy", "d MMMM yyyy",
]
data_convertida = F.coalesce(*[F.expr(f"try_to_date(`_data_txt`, '{fmt}')") for fmt in FORMATOS_DATA])
df = df.withColumn("data_lancamento", data_convertida)   # só vira NULL se NENHUM formato funcionou

# Passo 4: duração 
# Converte para inteiro; duração zero ou negativa não existe, então vira NULL.
df = converter_numero(df, "runtime", "_duracao", limpar_decimal, "int")

# Passo 5: seleção final (renomeando para português) 
silver_info_filmes = df.select(
    F.col("id").alias("id_filme"),
    limpar_texto("title").alias("titulo"),
    limpar_texto("original_title").alias("titulo_original"),
    F.col("data_lancamento"),
    F.when(F.col("_duracao") > 0, F.col("_duracao")).alias("duracao_minutos"),
    F.lower(limpar_texto("original_language")).alias("idioma_original"),
    status_traduzido.alias("status_filme"),
    limpar_texto("overview").alias("sinopse"),
    limpar_texto("tagline").alias("frase_divulgacao"),
    F.year("data_lancamento").alias("ano_lancamento"),        
)

salvar_silver(silver_info_filmes, "tb_info_filmes")

OK  silver.tb_info_filmes  |  97611 linhas


## 3. `silver.tb_financeiro_filmes`  (origem: `tb_movies_financials`)

**Passos:**
1. Deduplicar por filme.
2. Limpar `budget` e `revenue`: ausência textual → `NULL`, remover símbolos e separadores de milhar, converter para `DECIMAL(18,2)`.
3. Valor zerado, negativo ou absurdo → `NULL` (orçamento de US$ 0 significa "não sabemos", não "custou zero").
4. Converter para reais usando a cotação mais recente de `silver.tb_cotacao_dolar`.
5. Calcular lucro (USD e BRL) e margem de lucro %. 

> Decisão de negócio (taxa única): converti todos os filmes com a cotação mais recente disponível, ou seja,
> quanto valeria hoje em reais. Converter pela cotação da data de lançamento exigiria uma série histórica de dólar
> que o enunciado não fornece.
>
> Decisão de negócio (lucro): o lucro só é calculado quando a receita e o orçamento existem. Se tratássemos um
> orçamento ausente como 0, o lucro seria igual à receita inteira, o que é falso e distorceria os rankings.

In [0]:
df = ler_bronze("tb_movies_financials")

# Passos 1 e 2: chave, deduplicação e conversão
df = limpar_id(df, "id")
df = manter_versao_mais_recente(df, ["id"])

df = converter_numero(df, "budget",  "_orcamento", limpar_valor_monetario, "decimal(18,2)")
df = converter_numero(df, "revenue", "_receita",   limpar_valor_monetario, "decimal(18,2)")

# Passo 3: regra de negócio
def valor_valido(coluna: str):
    return F.when(F.col(coluna) > 0, F.col(coluna))

df = (
    df.withColumn("orcamento_usd", valor_valido("_orcamento"))
      .withColumn("receita_usd", valor_valido("_receita"))
)

# Passo 4: cotação
# Pega a cotação mais recente da Silver (já com forward fill, então nunca cai em fim de semana).
linha_cotacao = (
    spark.table(f"{BANCO_SILVER}.tb_cotacao_dolar")
    .filter(F.col("cotacao_compra").isNotNull())
    .orderBy(F.col("data_cotacao").desc())
    .first()
)
if linha_cotacao is None:
    print("ATENÇÃO: sem cotação disponível; colunas em BRL ficarão NULL.")
    taxa = F.lit(None).cast("decimal(18,6)")
else:
    print(f"Cotação usada: R$ {linha_cotacao['cotacao_compra']} (data {linha_cotacao['data_cotacao']})")
    taxa = F.lit(float(linha_cotacao["cotacao_compra"])).cast("decimal(18,6)")

df = (
    df.withColumn("orcamento_brl", (F.col("orcamento_usd") * taxa).cast("decimal(18,2)"))
      .withColumn("receita_brl", (F.col("receita_usd") * taxa).cast("decimal(18,2)"))
)

# Passo 5: lucro e margem
# Em SQL/Spark, NULL - 5 = NULL. Isso é o comportamento desejado: sem os dois lados não há lucro.
# A margem só é calculada se receita > 0 (isso evita divisão por zero).
lucro_usd = F.when(F.col("receita_usd").isNotNull() & F.col("orcamento_usd").isNotNull(),
                   F.col("receita_usd") - F.col("orcamento_usd"))
lucro_brl = F.when(F.col("receita_brl").isNotNull() & F.col("orcamento_brl").isNotNull(),
                   F.col("receita_brl") - F.col("orcamento_brl"))

df = df.withColumn("lucro_usd", lucro_usd).withColumn("lucro_brl", lucro_brl)
df = df.withColumn(
    "margem_lucro_percentual",
    F.when(F.col("lucro_usd").isNotNull() & (F.col("receita_usd") > 0),
           F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)).cast("decimal(18,2)"),
)

silver_financeiro = df.select(
    F.col("id").alias("id_filme"),
    "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual",
)
salvar_silver(silver_financeiro, "tb_financeiro_filmes")

Cotação usada: R$ 5.1111 (data 2026-09-21)
OK  silver.tb_financeiro_filmes  |  99006 linhas


## 4. `silver.tb_metricas_engajamento`  (origem: `tb_movies_metrics`)

Aqui aparece o Column Shift: por erro na origem, alguns valores saem de coluna, então uma coluna
de nota pode ter texto como "Drama". Na limpeza só aceitam texto com cara de número; o resto vira `NULL`.

Regras de negócio aplicadas depois da conversão:
- Nota TMDB e nota IMDb entre 0 e 10. Fora disso (inclusive nota multiplicada por erro de escala) fica = `NULL`. Não concerta a escala, porque não dá para ter certeza do valor original.
- Popularidade e contagens de votos não podem ser negativas então fica `NULL`.

In [0]:
df = ler_bronze("tb_movies_metrics")

df = limpar_id(df, "id")
df = manter_versao_mais_recente(df, ["id"])

# Conversão segura de tipos (texto sujo -> NULL, sem quebrar o pipeline)
df = converter_numero(df, "popularity",    "_popularidade", limpar_decimal, "double")
df = converter_numero(df, "vote_average",  "_nota_tmdb",    limpar_decimal, "double")
df = converter_numero(df, "vote_count",    "_votos_tmdb",   limpar_inteiro, "int")
df = converter_numero(df, "averageRating", "_nota_imdb",    limpar_decimal, "double")
df = converter_numero(df, "numVotes",      "_votos_imdb",   limpar_inteiro, "int")

def nota_valida(coluna: str):
    """between(0, 10) é inclusivo nas duas pontas; NULL continua NULL."""
    return F.when(F.col(coluna).between(0, 10), F.col(coluna))

def nao_negativo(coluna: str):
    return F.when(F.col(coluna) >= 0, F.col(coluna))

silver_metricas = df.select(
    F.col("id").alias("id_filme"),
    nao_negativo("_popularidade").alias("popularidade"),
    nota_valida("_nota_tmdb").alias("nota_media_tmdb"),
    nao_negativo("_votos_tmdb").alias("qtd_votos_tmdb"),
    nota_valida("_nota_imdb").alias("nota_media_imdb"),
    nao_negativo("_votos_imdb").alias("qtd_votos_imdb"),
)
salvar_silver(silver_metricas, "tb_metricas_engajamento")

OK  silver.tb_metricas_engajamento  |  95115 linhas


## 5. `silver.tb_avaliacoes_usuarios`  (origem: `tb_movies_reviews`)

- Nota fora de 0 a 10 → `NULL` (a avaliação continua existindo, só a nota inválida é descartada).
- Comentário vazio, `NULL` ou só espaços → texto padrão.
- Duplicados integrais (mesmo filme + usuário + nota + comentário) → mantém uma só linha.
  A deduplicação é feita depois da limpeza, para que "  bom " e "bom" contem como iguais.

In [0]:
df = ler_bronze("tb_movies_reviews")
df = limpar_id(df, "id")
df = converter_numero(df, "nota", "_nota", limpar_decimal, "double")

silver_avaliacoes = (
    df.select(
        F.col("id").alias("id_filme"),
        limpar_texto("nome").alias("nome_usuario"),
        F.when(F.col("_nota").between(0, 10), F.col("_nota")).alias("nota_usuario"),   # regra de negócio: 0 a 10
        F.coalesce(limpar_texto("comentario"), F.lit("Sem comentário")).alias("comentario_usuario"),
    )
    # dropDuplicates com a lista de colunas remove linhas idênticas naquela combinação
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)
salvar_silver(silver_avaliacoes, "tb_avaliacoes_usuarios")

OK  silver.tb_avaliacoes_usuarios  |  32412 linhas


## 6. `silver.tb_generos`  (origem: `tb_credits_and_tags`, coluna `genres`)

Explode = transformar uma célula com vários valores em várias linhas:

| id | genres |   →   | id | genero |
|---|---|---|---|---|
| 1 | Action, Drama |   | 1 | Action |
| | |   | 1 | Drama |

Sobre a sujeira, filtrei os pedaços que não parecem gênero:
- vazios e textos de ausência (Unknown, nan...);
- números puros;
- textos longos ou com muitas palavras (descrições);

In [0]:
creditos = manter_versao_mais_recente(
    limpar_id(ler_bronze("tb_credits_and_tags"), "id"),
    ["id"]
)

# Explode: 1 linha por gênero.
# Nulos e listas vazias não geram linhas.
generos = (
    creditos
    .select(
        F.col("id").alias("id_filme"),
        F.explode(dividir_lista("genres")).alias("genero_bruto")
    )
)

g = F.regexp_replace(
    F.trim(F.col("genero_bruto")),
    r"^['\s]+|['\s]+$",
    ""
)

parece_genero = (
    g.rlike(r"^[A-Za-zÀ-ÿ][A-Za-zÀ-ÿ '&\-]*$")
    & (F.length(g) <= 25)
    & (F.size(F.split(g, " ")) <= 3)
    & ~F.lower(g).isin(TOKENS_AUSENCIA)
)

silver_generos = (
    generos
    .filter(parece_genero)
    .select(
        "id_filme",
        F.initcap(g).alias("genero")
    )
    .dropDuplicates()
)

salvar_silver(silver_generos, "tb_generos")

OK  silver.tb_generos  |  142627 linhas


## 7. `silver.tb_pessoas_empresas`  (origem: `tb_credits_and_tags`)

Uma tabela única com **4 tipos** de entidade. Fazemos o mesmo *explode* de cada coluna e marcamos o tipo:

| Coluna de origem | `tipo_entidade` |
|---|---|
| `cast` | Ator |
| `directors` | Diretor |
| `writers` | Roteirista |
| `production_companies` | Produtora |

Detalhes:
- Capitalização padronizada com `initcap` ("CHRISTOPHER NOLAN" e "christopher nolan" viram "Christopher Nolan"), senão a deduplicação não reconheceria que é a mesma pessoa.
- Guardamos `posicao`: a ordem original do nome na lista. Como o elenco normalmente vem do papel mais importante para o menos importante, isso permite identificar os "atores principais" no Gold.
- Descartamos: nomes vazios, números puros, textos com mais de 100 caracteres (descrições deslocadas) e sobras como "Jr." ou "Inc." que surgem ao separar por vírgula.

In [0]:
ENTIDADES = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora",
}

# creditos já foi carregado e deduplicado na célula anterior; usamos de novo (uma linha por filme)
partes = []
for coluna_origem, tipo in ENTIDADES.items():
    # posexplode = explode que também devolve a posição (0, 1, 2...) de cada item na lista
    parte = creditos.select(
        F.col("id").alias("id_filme"),
        F.posexplode(dividir_lista(coluna_origem)).alias("posicao", "nome_bruto"),
    ).withColumn("tipo_entidade", F.lit(tipo))
    partes.append(parte)

# une as 4 tabelas em uma só (unionByName casa as colunas pelo NOME, não pela ordem)
entidades = partes[0]
for outra in partes[1:]:
    entidades = entidades.unionByName(outra)

nome = F.regexp_replace(F.trim(F.col("nome_bruto")), r"^['\s]+|['\s]+$", "")

nome_valido = (
    (F.length(nome) >= 2)
    & (F.length(nome) <= 100)                       # descarta textos descritivos deslocados
    & ~nome.rlike(r"^[\d\s.,\-]+$")                 # descarta números puros (ex.: nota que entrou na coluna)
    & ~F.lower(nome).isin(TOKENS_AUSENCIA)
    & ~F.lower(nome).isin(SUFIXOS_SOLTOS)
)

silver_pessoas_empresas = (
    entidades.filter(nome_valido)
    .select("id_filme", F.initcap(nome).alias("nome_entidade"), "tipo_entidade", "posicao")
    # Eliminar duplicados: se a mesma entidade aparece 2x no mesmo filme, fica 1 linha (com a menor posição)
    .groupBy("id_filme", "nome_entidade", "tipo_entidade")
    .agg(F.min("posicao").cast("int").alias("posicao"))
)
salvar_silver(silver_pessoas_empresas, "tb_pessoas_empresas")

OK  silver.tb_pessoas_empresas  |  890571 linhas


## 8. Validação final

Contagem de linhas, NULLs por coluna e alguns conferes rápidos.

In [0]:
tabelas_silver = [
    "tb_cotacao_dolar", "tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios", "tb_generos", "tb_pessoas_empresas",
]
for t in tabelas_silver:
    df_t = spark.table(f"{BANCO_SILVER}.{t}")
    print(f"\n=== {BANCO_SILVER}.{t} | {df_t.count()} linhas ===")
    resumo_nulos(df_t)


=== silver.tb_cotacao_dolar | 11 linhas ===


data_cotacao,cotacao_compra,cotacao_preenchida
0,0,0



=== silver.tb_info_filmes | 97611 linhas ===


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
0,0,0,2,10165,0,0,13615,74906,2



=== silver.tb_financeiro_filmes | 99006 linhas ===


...

id_filme,nome_usuario,nota_usuario,comentario_usuario
0,0,1651,0



=== silver.tb_generos | 142627 linhas ===


id_filme,genero
0,0



=== silver.tb_pessoas_empresas | 890571 linhas ===


id_filme,nome_entidade,tipo_entidade,posicao
0,0,0,0


In [0]:
# 1) Cada filme deve aparecer UMA vez em info/financeiro/métricas
for t in ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento"]:
    d = spark.table(f"{BANCO_SILVER}.{t}")
    print(t, "-> linhas:", d.count(), "| ids distintos:", d.select("id_filme").distinct().count())

# 2) Distribuição do status (deve ter só os 6 valores traduzidos + "Não Informado")
display(spark.table(f"{BANCO_SILVER}.tb_info_filmes").groupBy("status_filme").count().orderBy(F.col("count").desc()))

# 3) Gêneros encontrados (confira se todos fazem sentido)
display(spark.table(f"{BANCO_SILVER}.tb_generos").groupBy("genero").count().orderBy(F.col("count").desc()))

# 4) Quantidade de entidades por tipo
display(spark.table(f"{BANCO_SILVER}.tb_pessoas_empresas").groupBy("tipo_entidade").count())

tb_info_filmes -> linhas: 97611 | ids distintos: 97611
tb_financeiro_filmes -> linhas: 99006 | ids distintos: 99006
tb_metricas_engajamento -> linhas: 95115 | ids distintos: 95115


status_filme,count
Lançado,96261
Pós-Produção,697
Em Produção,604
Planejado,47
Não Informado,2


genero,count
Drama,32615
Documentary,19234
Comedy,18827
Thriller,10400
Horror,9833
Romance,7717
Action,6114
Crime,4786
Animation,4524
Tv Movie,4115


tipo_entidade,count
Ator,542536
Diretor,101508
Roteirista,126026
Produtora,120501
